# Colab — YOLOv8n × 3 seeds + 2 cross-country runs (T4)
**Runtime → Change runtime type → T4 GPU.** Then **Runtime → Run all**.

Checkpoints go to Google Drive (`MyDrive/road-defect/runs`). If Colab disconnects: reconnect and **Run all** again —
finished jobs are skipped and the running one resumes from its last epoch.
At the end `results_colab.zip` is also copied to `MyDrive/road-defect/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, shutil
REPO_URL = "https://github.com/AdonisYsh/road-defect-severity.git"   # <- your GitHub repo (must be public, or put a token in the URL)
ROOT = "/content/road-defect"
if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=True)
else:
    tmp = ROOT + "_clone"
    shutil.rmtree(tmp, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, tmp], check=True)
    os.makedirs(ROOT, exist_ok=True)
    shutil.copytree(tmp, ROOT, dirs_exist_ok=True)   # keeps anything the preflight left (data/raw, weights/)
    shutil.rmtree(tmp)
os.chdir(ROOT)
subprocess.run("pip install -q -r requirements-cloud.txt", shell=True, check=True)
print("repo ready at", os.getcwd())

In [ ]:
!python -m rdd.download && python -m rdd.prep

In [ ]:
!python -m rdd.jobs yolov8n_seed0 yolov8n_seed1 yolov8n_seed2

In [ ]:
!python -m rdd.jobs xc_india xc_japan

In [ ]:
!python -m rdd.benchmark

In [ ]:
!python -m rdd.report --pack
from google.colab import files
files.download('results_colab.zip')